# QuantJourney SDK - Domain Route Discovery and Contract Introspection

This notebook demonstrates a QuantJourney SDK workflow that uses domain discovery, aliases and route descriptions to inspect the governed API contract available to a tenant.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


In [ ]:
domains_list = qj.domains.list()
domain_tree = qj.domains.tree(scope='effective', include_aliases=True)
aliases = qj.domains.aliases()
route_names = ['equity.pricing.get_historical_prices', 'equity.fundamentals.get_financial_ratios_ttm', 'macro.economic.get_treasury_rates', 'derivatives.vol.get_vix_data', 'reference.identifiers.get_figi_data']
descriptions = {route: qj.domains.describe(route=route) for route in route_names}


In [ ]:
rows = []
for route, payload in descriptions.items():
    value = unwrap(payload)
    if isinstance(value, dict):
        rows.append({'route': route, 'domain': value.get('domain') or value.get('domain_path'), 'description': value.get('description'), 'required_scopes': value.get('required_scopes') or value.get('scopes'), 'connectors': value.get('connectors') or value.get('providers')})
    else:
        rows.append({'route': route, 'domain': None, 'description': None, 'required_scopes': None, 'connectors': None})
contract = pd.DataFrame(rows)
coverage = pd.Series({'domain_list_rows': len(as_rows(domains_list)), 'domain_tree_rows': len(as_rows(domain_tree)), 'aliases_rows': len(as_rows(aliases)), 'described_routes': contract['description'].notna().sum() if not contract.empty else 0})
display(coverage)
display(contract)


In [ ]:
plot_data = pd.Series({'list': len(as_rows(domains_list)), 'aliases': len(as_rows(aliases)), 'descriptions': len(descriptions), 'routes_checked': len(route_names)})
plot_data.plot(kind='bar', title='Domain discovery surface')
plt.ylabel('count')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.